In [3]:
import csv
import pandas as pd
import numpy as np
import math

data = [
    ['RollNo', 'Age', 'Income', 'Student', 'Credit Rating', 'Buys Computer'],
    [1, 'Young', 'High', 'No', 'Fair', 'No'],
    [2, 'Young', 'High', 'No', 'Excellent', 'No'],
    [3, 'Middle Aged', 'High', 'No', 'Fair', 'Yes'],
    [4, 'Senior', 'Medium', 'No', 'Fair', 'Yes'],
    [5, 'Senior', 'Low', 'Yes', 'Fair', 'Yes'],
    [6, 'Senior', 'Low', 'Yes', 'Excellent', 'No'],
    [7, 'Middle Aged', 'Low', 'Yes', 'Excellent', 'Yes'],
    [8, 'Young', 'Medium', 'No', 'Fair', 'No'],
    [9, 'Young', 'Low', 'Yes', 'Fair', 'Yes'],
    [10, 'Senior', 'Medium', 'Yes', 'Fair', 'Yes'],
    [11, 'Young', 'Medium', 'Yes', 'Excellent', 'Yes'],
    [12, 'Middle Aged', 'Medium', 'No', 'Excellent', 'Yes'],
    [13, 'Middle Aged', 'High', 'Yes', 'Fair', 'Yes'],
    [14, 'Senior', 'Medium', 'No', 'Excellent', 'No']
]

with open('studentdata.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(data)

df = pd.read_csv('studentdata.csv')
df = df.drop("RollNo", axis=1)

def find_entropy(df):
    target = df.columns[-1]
    values = df[target].unique()
    entropy = 0
    for value in values:
        p = len(df[df[target] == value]) / len(df)
        entropy -= p * math.log2(p)
    return entropy

def find_entropy_attribute(df, attribute):
    target = df.columns[-1]
    values = df[attribute].unique()
    total_entropy = 0
    for value in values:
        subset = df[df[attribute] == value]
        subset_entropy = 0
        target_values = subset[target].unique()
        for target_value in target_values:
            p = len(subset[subset[target] == target_value]) / len(subset)
            subset_entropy -= p * math.log2(p)
        weight = len(subset) / len(df)
        total_entropy += weight * subset_entropy
    return total_entropy

def find_winner(df):
    IG = []
    attributes = []
    entropy = find_entropy(df)

    for key in df.columns[:-1]:
        gain = entropy - find_entropy_attribute(df, key)
        IG.append(gain)
        attributes.append(key)
    return attributes[np.argmax(IG)]

def get_subtable(df, attribute, value):
    return df[df[attribute] == value].reset_index(drop=True)

def build_tree(df, tree=None):
    node = find_winner(df)
    att_values = np.unique(df[node])
    class_name = df.columns[-1]

    if tree is None:
        tree = {}
        tree[node] = {}

    for value in att_values:
        subtable = get_subtable(df, node, value)
        c1, counts = np.unique(subtable[class_name], return_counts=True)
        if len(counts) == 1:
            tree[node][value] = c1[0]
        else:
            tree[node][value] = build_tree(subtable)

    return tree

def predict_instance(inst, tree):
    for node in tree.keys():
        value = inst[node]
        tree = tree[node][value]
        if type(tree) is dict:
            return predict_instance(inst, tree)
        else:
            return tree

tree = build_tree(df)
print("ID3 Tree:")
print(tree)

predictions = []
for i in range(len(df)):
    inst = df.iloc[i, :]
    prediction = predict_instance(inst, tree)
    predictions.append(prediction)

print("Predictions:")
print(predictions)

from sklearn import metrics
print(metrics.classification_report(df.iloc[:, -1], predictions))

ID3 Tree:
{'Age': {'Middle Aged': 'Yes', 'Senior': {'Credit Rating': {'Excellent': 'No', 'Fair': 'Yes'}}, 'Young': {'Student': {'No': 'No', 'Yes': 'Yes'}}}}
Predictions:
['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
              precision    recall  f1-score   support

          No       1.00      1.00      1.00         5
         Yes       1.00      1.00      1.00         9

    accuracy                           1.00        14
   macro avg       1.00      1.00      1.00        14
weighted avg       1.00      1.00      1.00        14



In [4]:
import csv
import pandas as pd
import numpy as np
import math

data = [
    ['RollNo', 'Age', 'Income', 'Student', 'Credit Rating', 'Buys Computer'],
    [1, 'Young', 'High', 'No', 'Fair', 'No'],
    [2, 'Young', 'High', 'No', 'Excellent', 'No'],
    [3, 'Middle Aged', 'High', 'No', 'Fair', 'Yes'],
    [4, 'Senior', 'Medium', 'No', 'Fair', 'Yes'],
    [5, 'Senior', 'Low', 'Yes', 'Fair', 'Yes'],
    [6, 'Senior', 'Low', 'Yes', 'Excellent', 'No'],
    [7, 'Middle Aged', 'Low', 'Yes', 'Excellent', 'Yes'],
    [8, 'Young', 'Medium', 'No', 'Fair', 'No'],
    [9, 'Young', 'Low', 'Yes', 'Fair', 'Yes'],
    [10, 'Senior', 'Medium', 'Yes', 'Fair', 'Yes'],
    [11, 'Young', 'Medium', 'Yes', 'Excellent', 'Yes'],
    [12, 'Middle Aged', 'Medium', 'No', 'Excellent', 'Yes'],
    [13, 'Middle Aged', 'High', 'Yes', 'Fair', 'Yes'],
    [14, 'Senior', 'Medium', 'No', 'Excellent', 'No']
]

with open('studentdata.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(data)

df = pd.read_csv('studentdata.csv')

def findentropy(df):
    target = df.columns[-1]
    values = df[target].unique()
    entropy = 0
    for value in values:
        probability = len(df[df[target] == value]) / len(df)
        entropy -= probability * np.log2(probability)
    return entropy

def findentropyattribute(df, attribute):
    target = df.columns[-1]
    values = df[attribute].unique()
    totalentropy = 0
    for value in values:
        subset = df[df[attribute] == value]
        subsetentropy = 0
        targetvalues = subset[target].unique()
        for targetvalue in targetvalues:
            probability = len(subset[subset[target] == targetvalue]) / len(subset)
            subsetentropy -= probability * np.log2(probability)
        weight = len(subset) / len(df)
        totalentropy += weight * subsetentropy
    return totalentropy

def splitinformation(df, attribute):
    values = df[attribute].unique()
    splitinfo = 0
    for value in values:
        probability = len(df[df[attribute] == value]) / len(df)
        if probability > 0:
            splitinfo -= probability * np.log2(probability)
    return splitinfo

def gainratio(df, attribute):
    infogain = findentropy(df) - findentropyattribute(df, attribute)
    splitinfo = splitinformation(df, attribute)
    if splitinfo == 0:
        return 0
    return infogain / splitinfo

def findwinnerc45(df):
    GR = []
    attributes = []
    for key in df.columns[:-1]:
        if df[key].nunique() == len(df):
            continue
        GR.append(gainratio(df, key))
        attributes.append(key)
    return attributes[np.argmax(GR)]

def getsubtable(df, attribute, value):
    return df[df[attribute] == value].reset_index(drop=True)

def buildtreec45(df, tree=None):
    node = findwinnerc45(df)
    attvalue = np.unique(df[node])
    Class = df.columns[-1]

    if tree is None:
        tree = {}
        tree[node] = {}

    for value in attvalue:
        subtable = getsubtable(df, node, value)
        C1, counts = np.unique(subtable[Class], return_counts=True)
        if len(counts) == 1:
            tree[node][value] = C1[0]
        else:
            tree[node][value] = buildtreec45(subtable)

    return tree

def predictinst(inst, tree):
    for node in tree.keys():
        value = inst[node]
        tree = tree[node][value]
        if type(tree) is dict:
            return predictinst(inst, tree)
        else:
            return tree

treec45 = buildtreec45(df)
print("C4.5 Tree:")
print(treec45)

Y1label = []
for i in range(len(df)):
    inst = df.iloc[i, :]
    prediction = predictinst(inst, treec45)
    Y1label.append(prediction)

print("C4.5 Predictions:")
print(Y1label)

from sklearn import metrics
print(metrics.classification_report(df.iloc[:, -1], Y1label))

C4.5 Tree:
{'Age': {'Middle Aged': 'Yes', 'Senior': {'Credit Rating': {'Excellent': 'No', 'Fair': 'Yes'}}, 'Young': {'Student': {'No': 'No', 'Yes': 'Yes'}}}}
C4.5 Predictions:
['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
              precision    recall  f1-score   support

          No       1.00      1.00      1.00         5
         Yes       1.00      1.00      1.00         9

    accuracy                           1.00        14
   macro avg       1.00      1.00      1.00        14
weighted avg       1.00      1.00      1.00        14



In [5]:
import csv
import pandas as pd
import numpy as np
import math

data = [
    ['RollNo', 'Age', 'Income', 'Student', 'Credit Rating', 'Buys Computer'],
    [1, 'Young', 'High', 'No', 'Fair', 'No'],
    [2, 'Young', 'High', 'No', 'Excellent', 'No'],
    [3, 'Middle Aged', 'High', 'No', 'Fair', 'Yes'],
    [4, 'Senior', 'Medium', 'No', 'Fair', 'Yes'],
    [5, 'Senior', 'Low', 'Yes', 'Fair', 'Yes'],
    [6, 'Senior', 'Low', 'Yes', 'Excellent', 'No'],
    [7, 'Middle Aged', 'Low', 'Yes', 'Excellent', 'Yes'],
    [8, 'Young', 'Medium', 'No', 'Fair', 'No'],
    [9, 'Young', 'Low', 'Yes', 'Fair', 'Yes'],
    [10, 'Senior', 'Medium', 'Yes', 'Fair', 'Yes'],
    [11, 'Young', 'Medium', 'Yes', 'Excellent', 'Yes'],
    [12, 'Middle Aged', 'Medium', 'No', 'Excellent', 'Yes'],
    [13, 'Middle Aged', 'High', 'Yes', 'Fair', 'Yes'],
    [14, 'Senior', 'Medium', 'No', 'Excellent', 'No']
]

with open('studentdata.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(data)

df = pd.read_csv('studentdata.csv')

def findgini(df):
    target = df.columns[-1]
    values = df[target].unique()
    gini = 1.0
    for value in values:
        probability = len(df[df[target] == value]) / len(df)
        gini -= probability ** 2
    return gini

def findginiattribute(df, attribute):
    values = df[attribute].unique()
    totalgini = 0
    for value in values:
        subset = df[df[attribute] == value]
        subsetgini = findgini(subset)
        weight = len(subset) / len(df)
        totalgini += weight * subsetgini
    return totalgini

def findwinnercart(df):
    Giniscores = []
    attributes = []
    for key in df.columns[:-1]:
        if df[key].nunique() == len(df):
            continue
        Giniscores.append(findginiattribute(df, key))
        attributes.append(key)
    return attributes[np.argmin(Giniscores)]

def getsubtable(df, attribute, value):
    return df[df[attribute] == value].reset_index(drop=True)

def buildtreecart(df, tree=None):
    node = findwinnercart(df)
    attvalue = np.unique(df[node])
    Class = df.columns[-1]

    if tree is None:
        tree = {}
        tree[node] = {}

    for value in attvalue:
        subtable = getsubtable(df, node, value)
        C1, counts = np.unique(subtable[Class], return_counts=True)
        if len(counts) == 1:
            tree[node][value] = C1[0]
        else:
            tree[node][value] = buildtreecart(subtable)

    return tree

def predictinst(inst, tree):
    for node in tree.keys():
        value = inst[node]
        tree = tree[node][value]
        if type(tree) is dict:
            return predictinst(inst, tree)
        else:
            return tree

treecart = buildtreecart(df)
print("CART Tree:")
print(treecart)

Ycartlabel = []
for i in range(len(df)):
    inst = df.iloc[i, :]
    prediction = predictinst(inst, treecart)
    Ycartlabel.append(prediction)

print("CART Predictions:")
print(Ycartlabel)

from sklearn import metrics
print(metrics.classification_report(df.iloc[:, -1], Ycartlabel))

CART Tree:
{'Age': {'Middle Aged': 'Yes', 'Senior': {'Credit Rating': {'Excellent': 'No', 'Fair': 'Yes'}}, 'Young': {'Student': {'No': 'No', 'Yes': 'Yes'}}}}
CART Predictions:
['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
              precision    recall  f1-score   support

          No       1.00      1.00      1.00         5
         Yes       1.00      1.00      1.00         9

    accuracy                           1.00        14
   macro avg       1.00      1.00      1.00        14
weighted avg       1.00      1.00      1.00        14

